# Article Classifier

In [10]:
#imports
import pandas as pd
from tqdm import tqdm
from nltk.tokenize import RegexpTokenizer
import re
from IPython.display import display, HTML
import os

folder_path = 'election_programs_parsed.csv'

In [3]:
# AI-related keywords (regex-style, separated by "|")
keywords = (
    "kunstmatige intelligentie|artificial intelligence|artificiële intelligentie|AI|generatieve AI|"
    "generatieve kunstmatige intelligentie|generatieve artificiële intelligentie|"
    "machine learning|machinaal leren|diep leren|deep learning|neurale netwerken|"
    "large language model|grote taalmodel*|LLM|chatbot*|GPT|ChatGPT|Bard|Claude|"
    "Gemini|LLaMA|openai|kunstmatige intelligentie systeem*|intelligente algoritme*|"
    "slimme algoritme*|automatische besluitvorming|automatisch beslissysteem|"
    "algoritmische besluitvorming|algoritme*|cognitieve technologie*|AI-technologie*|"
    "AI-systeem*|AI-toepassing*|AI-model*|spraakherkenning|beeldherkenning|"
    "computer vision|natuurlijke taalverwerking|natural language processing|NLP"
)

# Convert into a list and a set
keywords = keywords.strip().split('|')
set_ai_words = set(keywords)


In [15]:
df = pd.read_csv(folder_path, index_col=0)
df

,party,year,body
0,BIJ1,2025,Doe eerlijk. Doe eerlijk. STEM Tweede Kamer ve...
1,50PLUS,2025,Verkiezingsprogramma 2025 - 2029 1 Verkiezings...
2,50PLUS,2019,1 Onze toekomst in Europa. Solidair met jong e...
3,GroenLinks,2019,Resolution: Freedom opportunity prosperity: th...
4,DENK,2018,LIJST 1 SAMEN MAKEN WIJ ER WERK VAN Vooraf Eco...
...,...,...,...
211,DENK,2025,1 VRIJ VERBOND VERKIEZINGS- PROGRAMMA 2025 – 2...
212,VVD,2019,PROGRAMMACOMMISSIE Ruben Brekelmans Debbie van...
213,DENK,2017,Concept verkiezingsprogramma 2017-2021 ZEKER N...
214,DENK,2017,VVD verkiezingsprogramma 2017-2021 ZEKER NEDER...


In [16]:
# Pattern that detects any of the keywords (safe + exact match)
_kw_list = [k for k in set_ai_words if isinstance(k, str) and k.strip()]
_kw_list = sorted(set(_kw_list), key=len, reverse=True)

_AI_PAT = re.compile(
    r'\b(' + '|'.join(re.escape(k) for k in _kw_list) + r')\b',
    re.IGNORECASE
)

# Weiwei filter
_WEIWEI_PAT = re.compile(r'\bweiwei\b', re.IGNORECASE)

# Pattern that detects the combined form "kunstmatige intelligentie (AI)" ---
_KI_AI_PAT = re.compile(r'kunstmatige\s+intelligentie\s*\(\s*ai\s*\)', re.IGNORECASE)

def _collapse_ki_ai(matches: list[str], text: str) -> list[str]:
    """
    If the text contains 'kunstmatige intelligentie (AI)', remove up to that many 'AI'
    occurrences from the match list so the pair counts as ONE hit.
    """
    n_pairs = len(_KI_AI_PAT.findall(text))
    if n_pairs == 0:
        return matches
    kept, removed = [], 0
    for m in matches:
        if m.lower() == "ai" and removed < n_pairs:
            removed += 1         # drop this 'AI' because it's part of the pair
        else:
            kept.append(m)
    return kept

# --- 3) Classification ---
def ai_classification(df, title_col='party', body_col='body'):
    labels, matched_title, matched_body, matched_all = [], [], [], []
    n_hits_title_total, n_hits_body_total = [], []

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        title = row[title_col] if pd.notna(row[title_col]) else ""
        body  = row[body_col]  if pd.notna(row[body_col])  else ""

        # regex matches
        title_matches = _AI_PAT.findall(str(title))
        body_matches  = _AI_PAT.findall(str(body))
        
        # --- filter: remove 'weiwei' matches ---
        title_matches = [m for m in title_matches if not _WEIWEI_PAT.findall(m)]
        body_matches  = [m for m in body_matches  if not _WEIWEI_PAT.findall(m)]

        # --- collapse KI (AI) double-counts ---
        title_matches = _collapse_ki_ai(title_matches, title)
        body_matches  = _collapse_ki_ai(body_matches, body)

        # --- rule: ignore 'claude' if it's the ONLY match across title+body ---
        all_lower = [m.lower() for m in (title_matches + body_matches)]
        if set(all_lower) == {"claude"}:
            title_matches, body_matches = [], []

        # decision rule
        title_match = len(title_matches) >= 1
        body_match  = len(body_matches)  >= 2
        label = "yes" if (title_match or body_match) else "no"

        labels.append(label)
        matched_title.append(sorted(set(m.lower() for m in title_matches)))
        matched_body.append(sorted(set(m.lower() for m in body_matches)))
        matched_all.append(sorted(set(m.lower() for m in (title_matches + body_matches))))
        n_hits_title_total.append(len(title_matches))
        n_hits_body_total.append(len(body_matches))

    # write back
    df['ai_related'] = labels
    df['matched_keywords_title'] = matched_title
    df['matched_keywords_body']  = matched_body
    df['matched_keywords_all']   = matched_all
    df['n_hits_title_total'] = n_hits_title_total
    df['n_hits_body_total']  = n_hits_body_total
    return labels


In [17]:

#apply the classification to the data
df = pd.read_csv(folder_path, index_col=0)

clabels = ai_classification(df)  # populates new columns on df

#check the counts of climate-related articles
print(df['ai_related'].value_counts())

#function to highlight keywords in the text (safer: escape keywords, handle None)
def highlight_keywords(text, keywords):
    if text is None:
        return ""
    out = str(text)
    for keyword in keywords:
        pat = rf'\b{re.escape(keyword)}\b'
        out = re.sub(pat, f'<b style="font-size: larger;">{keyword}</b>', out, flags=re.IGNORECASE)
    return out




100%|██████████| 216/216 [00:05<00:00, 42.03it/s]

ai_related
no     153
yes     63
Name: count, dtype: int64


In [18]:
df

,party,year,body,ai_related,matched_keywords_title,matched_keywords_body,matched_keywords_all,n_hits_title_total,n_hits_body_total
0,BIJ1,2025,Doe eerlijk. Doe eerlijk. STEM Tweede Kamer ve...,no,[],[ai],[ai],0,1
1,50PLUS,2025,Verkiezingsprogramma 2025 - 2029 1 Verkiezings...,yes,[],"[ai, artificiële intelligentie, kunstmatige in...","[ai, artificiële intelligentie, kunstmatige in...",0,4
2,50PLUS,2019,1 Onze toekomst in Europa. Solidair met jong e...,no,[],[],[],0,0
3,GroenLinks,2019,Resolution: Freedom opportunity prosperity: th...,yes,[],[artificial intelligence],[artificial intelligence],0,2
4,DENK,2018,LIJST 1 SAMEN MAKEN WIJ ER WERK VAN Vooraf Eco...,no,[],[],[],0,0
...,...,...,...,...,...,...,...,...,...
211,DENK,2025,1 VRIJ VERBOND VERKIEZINGS- PROGRAMMA 2025 – 2...,no,[],[],[],0,0
212,VVD,2019,PROGRAMMACOMMISSIE Ruben Brekelmans Debbie van...,yes,[],[kunstmatige intelligentie],[kunstmatige intelligentie],0,6
213,DENK,2017,Concept verkiezingsprogramma 2017-2021 ZEKER N...,no,[],[],[],0,0
214,DENK,2017,VVD verkiezingsprogramma 2017-2021 ZEKER NEDER...,no,[],[kunstmatige intelligentie],[kunstmatige intelligentie],0,1


In [8]:
#title_to_find = "De stad die muziek ademt De stad die muziek ademt	TR"

#row = df[df['title'] == title_to_find]
#row


In [29]:
import re
import pandas as pd
from IPython.display import display, HTML

def extract_highlighted_snippets(text, keywords, window=5, max_snippets=5):
    """
    Return HTML with up to `max_snippets` snippets from `text`, each containing a
    matched keyword and `window` words of context on both sides.
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    # Build regex for keywords
    pattern = r'\b(?:' + '|'.join(re.escape(w) for w in keywords) + r')\b'
    flags = re.IGNORECASE

    # All keyword matches (as character spans)
    matches = list(re.finditer(pattern, text, flags))
    if not matches:
        return ""

    # Precompute word spans to translate char positions -> word indices
    word_spans = [(m.start(), m.end()) for m in re.finditer(r'\S+', text)]

    def word_index_for_pos(pos):
        # Small linear search is fine for typical lengths; could be bisected if huge
        for i, (s, e) in enumerate(word_spans):
            if s <= pos < e:
                return i
        return None

    # Build context windows (as char spans), merging overlaps
    windows = []
    for m in matches:
        wi = word_index_for_pos(m.start())
        if wi is None:
            continue
        start_w = max(0, wi - window)
        end_w   = min(len(word_spans), wi + window + 1)
        start_c = word_spans[start_w][0]
        end_c   = word_spans[end_w - 1][1]

        if windows and start_c <= windows[-1][1]:
            # merge with previous overlapping window
            windows[-1] = (windows[-1][0], max(windows[-1][1], end_c))
        else:
            windows.append((start_c, end_c))

    # Limit number of snippets
    windows = windows[:max_snippets]

    # Build HTML snippets with highlighting
    pieces = []
    for s, e in windows:
        chunk = text[s:e]
        chunk = re.sub(pattern,
                       lambda mm: '<span style="font-weight:bold;color:green;">{}</span>'.format(mm.group(0)),
                       chunk, flags=flags)
        prefix = '...' if s > 0 else ''
        suffix = '...' if e < len(text) else ''
        pieces.append(prefix + chunk + suffix)

    return ' '.join(pieces)


# --- Keep your original highlighter for titles (optional) ---
def highlight_keywords(text, keywords):
    if not isinstance(keywords, (set, list)):
        raise ValueError("Keywords must be a set or list")
    pattern = r'\b(?:' + '|'.join(re.escape(word) for word in keywords) + r')\b'
    return re.sub(
        pattern,
        lambda m: f'<span style="font-size: 1.5em; font-weight: bold; color: green;">{m.group(0)}</span>',
        text or "",
        flags=re.IGNORECASE
    )


def inspect_ai_related(df, keywords, num_samples=5, window=5, max_snippets=5):
    """
    Displays random samples of AI-related rows with keyword-highlighted snippets.
    """
    subset = df[df['ai_related'] == 'yes']
    if subset.empty:
        display(HTML("<p><em>No AI-related rows found.</em></p>"))
        return

    n = min(len(subset), num_samples)
    samples = subset.sample(n=n, random_state=40)

    for _, row in samples.iterrows():
        title = row.get('party') or row.get('title') or ''
        body  = row.get('body')  or ''

        highlighted_title = highlight_keywords(str(title), keywords)
        # Only show short snippets around matches in the body
        highlighted_body_snippets = extract_highlighted_snippets(str(body), keywords,
                                                                 window=20,
                                                                 max_snippets=max_snippets)

        display(HTML(f'<h2>Title: {highlighted_title}</h2>'))
        if highlighted_body_snippets:
            display(HTML(f'<p>{highlighted_body_snippets}</p>'))
        else:
            display(HTML('<p><em>No keyword hits in body.</em></p>'))
        display(HTML('<hr>'))


In [20]:
df['ai_related'].value_counts()


ai_related
no     153
yes     63
Name: count, dtype: int64

In [30]:
inspect_ai_related(df, keywords)

In [ ]:
##create random sample of X articles
#n_per_outlet = 30 // df['outlet'].nunique()

#balanced_sample1 = (
#    df.groupby('outlet', group_keys=False)
#      .apply(lambda x: x.sample(n=n_per_outlet, random_state=42))
#      .sample(frac=1, random_state=42)  # shuffle
#      .reset_index(drop=True)
#)

In [ ]:
#balanced_sample1.to_csv('sample.csv')

In [22]:
# Save the classified data
df.to_csv('party_programs_classified.csv')